# **Triển khai dự án Data Science - Project 2 - Content-based Recommendation**


#

---



## **Business Understanding**
* **Bối cảnh:** Dữ liệu nhà ở được đăng bán trên Nhà Tốt.
* **Xác định vấn đề:**
  * *Mục tiêu/ vấn đề:*
    * Xây dựng hệ thống gợi ý nhà ở tương tự cho người mua.
    * Áp dụng các kỹ thuật đề xuất dựa trên nội dung (Content-based) để gợi ý các căn nhà có đặc điểm tương đồng với căn nhà mà người dùng đang xem.
  * *Xây dựng mô hình:*
    * Content-based filtering:
      * cosine_similarity.
      * Gensim.
    * Hybrid Recommendation.
      * Content Similarity (từ Content-based filtering)
      * Price Similarity.
      * Location Similarity.



---



# **Import thư viện**

In [3]:
# !pip install gensim
# !pip install pyvi
# !pip install beautifulsoup4
# !pip install ydata_profiling

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from ydata_profiling import ProfileReport

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity, pairwise_distances
from sklearn.preprocessing import MinMaxScaler
# from underthesea import word_tokenize, pos_tag, sent_tokenize
from pyvi.ViTokenizer import tokenize
from gensim import corpora, models, similarities
import re
from bs4 import BeautifulSoup
import joblib


from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)
import warnings
warnings.filterwarnings('ignore')

#


---



# **Đọc dữ liệu**

In [5]:
cd D:\ĐÔNG HY\Tai_xuong\Programs\Hoc_code\DL_ON_TANG_DONG_HY\Khoa7_Do_an_tot_nghiep_DS\nhatot_recommender_and_clustering

D:\ĐÔNG HY\Tai_xuong\Programs\Hoc_code\DL_ON_TANG_DONG_HY\Khoa7_Do_an_tot_nghiep_DS\nhatot_recommender_and_clustering


In [6]:
# Đọc dữ liệu ở quận Bình Thạnh
df_bt=pd.read_csv('Data/quan-binh-thanh.csv')
df_bt['quan']='Bình Thạnh'

# Đọc dữ liệu ở quận Gò Vấp
df_gv = pd.read_csv('Data/quan-go-vap.csv')
df_gv['quan']='Gò Vấp'

# Đọc dữ liệu ở quận Phú Nhuận
df_pn = pd.read_csv('Data/quan-phu-nhuan.csv')
df_pn['quan']='Phú Nhuận'

In [7]:
# Gộp thành 1 DF duy nhất
df_raw=pd.concat([df_bt,df_gv,df_pn], ignore_index=True)
print(f"Kích thước dữ liệu sau khi gộp: {df_raw.shape}")

Kích thước dữ liệu sau khi gộp: (8273, 24)


In [8]:
STOP_WORD_FILE = 'Data/files/vietnamese-stopwords.txt'

In [9]:
with open(STOP_WORD_FILE, 'r', encoding='utf-8') as file:
    stop_words = file.read()

stop_words = stop_words.split('\n')

In [10]:
df_raw.head(10)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [11]:
df_raw.columns

Index(['tieu_de', 'gia_ban', 'don_gia', 'dien_tich', 'dia_chi', 'mo_ta',
       'dien_thoai', 'loai_hinh', 'dien_tich_dat', 'dien_tich_su_dung',
       'gia_m2', 'giay_to_phap_ly', 'so_phong_ngu', 'so_phong_ve_sinh',
       'tong_so_tang', 'tinh_trang_noi_that', 'huong_cua_chinh', 'dac_diem',
       'chieu_ngang', 'chieu_dai', 'ma_can', 'ten_phan_khu_lo', 'bieu_do_gia',
       'quan'],
      dtype='object')

#


---



# **Data Understanding/ Acquire**
Dữ liệu được cung cấp sẵn gồm có 3 tập tin:

    quan-binh-thanh.csv,
    quan-go-vap.csv,
    quan-phu-nhuan.csv.


Các tập tin đã được tổng hợp lại, tạo thành bộ dữ liệu chứa thông tin về các căn nhà riêng lẻ đang được rao bán tại các Quận Bình Thạnh, Phú Nhuận, Gò Vấp ở Thành phố Hồ Chí Minh trên nền tảng Nhà Tốt.


Các thuộc tính trong bộ dữ liệu bao gồm:
|Tên cột|Ý nghĩa|
|-------|-------|
|tieu_de| Tiêu đề của bài đăng rao bán nhà.|
|gia_ban| Tổng giá bán của bất động sản (ví dụ: tỷ, triệu).|
|don_gia| Mức giá tính trên mỗi mét vuông.|
|dien_tich| Tổng diện tích sử dụng hoặc diện tích công nhận.|
|dia_chi| Địa chỉ chi tiết của căn nhà (bao gồm tên đường, phường và quận).|
|mo_ta| Nội dung mô tả chi tiết về đặc điểm, kết cấu, vị trí và các tiện ích xung quanh căn nhà.|
|dien_thoai| Số điện thoại liên hệ của người đăng tin.|
|loai_hinh| Loại hình bất động sản (ví dụ: Nhà ngõ, hẻm).|
|dien_tich_dat| Diện tích mặt bằng đất thực tế.|
|dien_tich_su_dung| Tổng diện tích sàn sử dụng của căn nhà.|
|gia_m2| Giá bán quy đổi ra đơn vị triệu/m2.|
|giay_to_phap_ly| Tình trạng giấy tờ của bất động sản (ví dụ: Đã có sổ).|
|so_phong_ngu| Số lượng phòng ngủ có trong nhà.|
|so_phong_ve_sinh| Số lượng nhà vệ sinh/WC.|
|tong_so_tang| Tổng số tầng của căn nhà.|
|tinh_trang_noi_that| Tình trạng bàn giao nội thất (ví dụ: Nội thất đầy đủ, hoàn thiện cơ bản).|
|huong_cua_chinh| Hướng chính của ngôi nhà.|
|dac_diem| Các đặc điểm nổi bật khác (ví dụ: Nhà nở hậu, hẻm xe hơi).|
|chieu_ngang| Chiều rộng mặt tiền của căn nhà.|
|chieu_dai| Chiều sâu/chiều dài của căn nhà.|
|ma_can| Mã số định danh của căn nhà (nếu có).|
|ten_phan_khu_lo| Tên phân khu hoặc lô đất (nếu có).|
|bieu_do_gia| Dữ liệu lịch sử biến động giá tại khu vực xung quanh trong 12 tháng tính đến thời điểm thu thập dữ liệu.|

#


---



# **EDA**

In [12]:
# # Thực hiện EDA bằng ProfileReport
# profile = ProfileReport(df_raw, title="Báo cáo Phân tích Dữ liệu Nhà Tốt", minimal=True)
# profile.to_notebook_iframe()

In [13]:
# # Trực quan hóa 
# # Thiết lập giao diện
# sns.set(style="whitegrid")
# fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# # 1. Số lượng bài đăng theo Quận
# sns.countplot(data=df_raw, x='quan', ax=axes[0, 0], palette='viridis')
# axes[0, 0].set_title('Phân bố bài đăng theo Quận')
# axes[0, 0].tick_params(axis='x', rotation=45)

# # 2. Phân loại hình bất động sản
# sns.countplot(data=df_raw, y='loai_hinh', ax=axes[0, 1], palette='magma', order=df_raw['loai_hinh'].value_counts().index)
# axes[0, 1].set_title('Cơ cấu Loại hình nhà ở')

# # 3. Tình trạng Giấy tờ pháp lý
# legal_counts = df_raw['giay_to_phap_ly'].value_counts().head(5)
# sns.barplot(x=legal_counts.values, y=legal_counts.index, ax=axes[1, 0], palette='rocket')
# axes[1, 0].set_title('Top Tình trạng Pháp lý')

# # 4. Phân bố Giá m2 (Giá này thường là dạng chuỗi nén cần xử lý nhanh để xem trend)
# df_raw['gia_m2_num'] = df_raw['gia_m2'].str.extract('(\d+)').astype(float)
# sns.boxplot(data=df_raw, x='quan', y='gia_m2_num', ax=axes[1, 1], palette='Set2')
# axes[1, 1].set_title('Biến động đơn giá (triệu/m2) theo Quận')
# axes[1, 1].set_ylim(0, 500) # Giỗi hạn để tránh outlier quá lớn

# plt.tight_layout()
# plt.show()

#

---



# **Data Preprocessing**

In [14]:
df_1=df_raw.copy()

In [15]:
pd.set_option('display.max_colwidth', None)
# Do cần để ý text nên mình muốn xem full text ở các hàng.

## **Làm sạch dữ liệu HTML và icon**





In [16]:
def clean_html_icon(text):
    if pd.isna(text):
        return ""

    # 1. Loại bỏ các thẻ HTML
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text()

    # 2. Loại bỏ Emojis và các ký tự biểu tượng đặc biệt
    # Cách này sử dụng Regex để giữ lại các ký tự chữ cái, số, dấu câu cơ bản và khoảng trắng
    # Loại bỏ các ký tự có dải Unicode đặc biệt (thường là icon/emoji)
    text = re.sub(r'[^\x00-\x7F\u00C0-\u1EF9]+', ' ', text)

    # 3. Làm sạch khoảng trắng và xuống dòng
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text)

    # Xóa sđt (pattern cơ bản cho SĐT Việt Nam)
    text = re.sub(r'(0[3|5|7|8|9])+([0-9]{8})', ' ', text)

    return text.strip().lower()

df_1["mo_ta_clean"] = df_1["mo_ta"].apply(clean_html_icon)

In [17]:
df_1[["mo_ta_clean"]].head(10)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


## **Tạo nội dung tổng hợp - Feature engineering**

In [18]:
# Danh sách các cột text quan trọng
important_cols = [
    'loai_hinh', 'tinh_trang_noi_that',
    'huong_cua_chinh', 'dac_diem'
]

***Lý do chọn:***
* `loai_hinh`: Phân loại cốt lõi (Nhà phố, hẻm, mặt tiền).
* `tinh_trang_noi_that`: "Đầy đủ" hay "nhà thô" là những đặc điểm rất tương đồng.
* `huong_cua_chinh`: Một người thích hướng Đông sẽ muốn xem các căn tương tự.
* `dac_diem`: Những điểm nổi bật khác.
* Ngoài ra còn `mo_ta_clean`: Phần nội dung chi tiết còn lại (gộp luôn chung với các cột kia ở function bên dưới).

In [19]:
# Hàm để gộp các cột, xử lý luôn giá trị NaN để không bị lỗi
def create_content(row):
    # Lấy các cột quan trọng, chuyển về string và bỏ qua NaN
    parts = [str(row[col]) for col in important_cols if pd.notna(row[col])]
    # Cộng thêm phần mô tả đã làm sạch ở cuối
    if pd.notna(row['mo_ta_clean']):
        parts.append(row['mo_ta_clean'])

    return " ".join(parts)

# Áp dụng tạo cột Content mới
df_1['Content'] = df_1.apply(create_content, axis=1)

In [20]:
df_1[['Content']].head(10)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [21]:
# word_tokenize
df_1["Content_wt"]=df_1['Content'].apply(lambda x: tokenize(x))

In [22]:
df_1[["Content", "Content_wt"]].head(20)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [23]:
df_1.head(10)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [24]:
# Tạo cột id chạy từ 0 đến hết dataframe
df_1['id'] = range(len(df_1))

# Đưa cột id lên đầu bảng cho dễ nhìn
cols = ['id'] + [col for col in df_1.columns if col != 'id']
df_1 = df_1[cols]

In [25]:
# Chỉ lấy giá trị phường (ví dụ: "7", "25", "Vạn Thạnh")
df_1['phuong_val'] = df_1['dia_chi'].str.extract(r'Phường\s+([\w\d]+)')

In [26]:
# Làm sạch cột giá
def clean_price(price_str):
    if pd.isna(price_str) or not isinstance(price_str, str):
        return None

    # Chuyển về chữ thường, đổi dấu phẩy thành dấu chấm
    s = price_str.lower().replace(',', '.').strip()

    if 'tỷ' in s:
        # Lấy số, bỏ chữ 'tỷ'
        val = float(s.replace('tỷ', '').strip())
        return val
    elif 'triệu' in s:
        # Lấy số, bỏ chữ 'triệu' và chia cho 1000 để về đơn vị Tỷ
        val = float(s.replace('triệu', '').strip())
        return val / 1000
    return None

# Áp dụng hàm cho cột gia_ban
df_1['gia_ban_ty'] = df_1['gia_ban'].apply(clean_price)

In [27]:
df_1.columns

Index(['id', 'tieu_de', 'gia_ban', 'don_gia', 'dien_tich', 'dia_chi', 'mo_ta',
       'dien_thoai', 'loai_hinh', 'dien_tich_dat', 'dien_tich_su_dung',
       'gia_m2', 'giay_to_phap_ly', 'so_phong_ngu', 'so_phong_ve_sinh',
       'tong_so_tang', 'tinh_trang_noi_that', 'huong_cua_chinh', 'dac_diem',
       'chieu_ngang', 'chieu_dai', 'ma_can', 'ten_phan_khu_lo', 'bieu_do_gia',
       'quan', 'mo_ta_clean', 'Content', 'Content_wt', 'phuong_val',
       'gia_ban_ty'],
      dtype='object')

In [28]:
col_drop=['tieu_de', 'gia_ban', 'don_gia', 'dien_tich', 'dia_chi', 'mo_ta',
           'dien_thoai', 'loai_hinh', 'dien_tich_dat', 'dien_tich_su_dung',
           'gia_m2', 'giay_to_phap_ly', 'so_phong_ngu', 'so_phong_ve_sinh',
           'tong_so_tang', 'tinh_trang_noi_that', 'huong_cua_chinh', 'dac_diem',
           'chieu_ngang', 'chieu_dai', 'ma_can', 'ten_phan_khu_lo', 'bieu_do_gia','Content']
df_1.drop(columns=col_drop, inplace=True)

## **Handle missing values**

In [29]:
df_1=df_1.dropna(subset=['mo_ta_clean']).reset_index(drop=True)

## **Handle duplicated values**

In [30]:
df_1=df_1.drop_duplicates(subset=['mo_ta_clean']).reset_index(drop=True)

## **Preprocessed Data**

In [31]:
df_1.head(20)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [32]:
# Save lại file đã làm sạch
df_1.to_csv('Data/cleaned_data_recommend.csv', index=False)


#

---



# **Cosine Similarity**

## **Preparation**

In [33]:
# Vector hóa nội dung
vectorizer = TfidfVectorizer(
    analyzer='word',
    stop_words=stop_words,
    max_df=0.8, # loại bớt các từ quá phổ biến
    min_df=2,  # Loại bỏ những từ chỉ xuất hiện đúng 1 lần -> giảm nhiễu
    max_features=10000,
    token_pattern=r'(?u)\b\w+\b' # Tiếng Việt
)
tfidf_matrix = vectorizer.fit_transform(df_1['Content_wt'])

# Tính toán độ tương đồng
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [34]:
df_show = pd.DataFrame(cosine_sim)
# df_show

## **Hàm đề xuất nhà**

In [35]:
def display_recommendations_cosine(house_id, cosine_sim, df, nums=5):
    # 1. Kiểm tra ID và lấy index (Giữ nguyên logic cũ)
    if house_id not in df['id'].values:
        print(f"ID {house_id} không tồn tại.")
        return

    idx = df.index[df['id'] == house_id][0]
    sim_scores = cosine_sim[idx]
    
    # 2. Tìm Top N (Giữ nguyên logic NumPy nhanh)
    top_idx_partition = np.argpartition(sim_scores, -(nums + 1))[-(nums + 1):]
    top_idx_sorted = top_idx_partition[np.argsort(sim_scores[top_idx_partition])][::-1]
    house_indices = [i for i in top_idx_sorted if i != idx][:nums]
    scores = sim_scores[house_indices]

    # 3. Hiển thị thông tin nhà hiện tại
    print(f"--- ĐANG XEM NHÀ (ID: {house_id}) ---")
    current_house = df.iloc[[idx]][['id', 'quan', 'mo_ta_clean']].copy()
    current_house.columns = ['ID', 'Quận', 'Mô tả']
    display(current_house)

    print("\n", "-"*30, f"TOP {nums} NHÀ TƯƠNG TỰ", "-"*30)

    # 4. Tạo DataFrame kết quả và Format thủ công
    recommendations = df.iloc[house_indices][['id', 'quan', 'mo_ta_clean']].copy()
    
    # Chuyển đổi score thành dạng text có dấu % (Ví dụ: 95.50%)
    recommendations['Độ tương đồng'] = [f"{s*100:.2f}%" for s in scores]
    
    # Đổi tên cột sang tiếng Việt cho chuyên nghiệp
    recommendations.columns = ['ID', 'Quận', 'Mô tả', 'Độ tương đồng']
    
    # Hiển thị bảng (Lúc này là bảng chuẩn của VS Code/Jupyter)
    display(recommendations.reset_index(drop=True))

## **Testing**

In [36]:
# Chạy thử nghiệm với house_id = 1
display_recommendations_cosine(house_id=1, cosine_sim=cosine_sim, df=df_1)

--- ĐANG XEM NHÀ (ID: 1) ---


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



 ------------------------------ TOP 5 NHÀ TƯƠNG TỰ ------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


***Nhận xét:***
* *Đặc điểm gốc:* Nhà ở khu vực giáp Bình Thạnh, Gò Vấp, diện tích 62m2, 4 tầng, 4PN, nội thất đầy đủ.

* *Kết quả gợi ý:*
  * Độ chính xác về tiện ích: Nhìn có vẻ tốt. Căn gợi ý số 1 (`id=2638`) và số 2 (`id=6721`) đều có các từ khóa như `"nhà mới", "full nội thất", "tầng", "PN"`.
  * Độ tương đồng (Similarity Score): Khá cao, đạt từ 44% đến 58%.

In [37]:
# Chạy thử nghiệm với house_id = 10
display_recommendations_cosine(house_id=10, cosine_sim=cosine_sim, df=df_1)

--- ĐANG XEM NHÀ (ID: 10) ---


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



 ------------------------------ TOP 5 NHÀ TƯƠNG TỰ ------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


***Nhận xét:***
* *Đặc điểm gốc:* Nhà ở Bình Thạnh (gần Chợ Bà Chiểu, BV Gia Định), hẻm xe tải, nở hậu, 4PN.
* *Kết quả gợi ý:*
  * Độ chính xác về Vị trí: Tốt. Các căn gợi ý như `id=1990` và `id=1211` đều tập trung đúng khu vực `"Chợ Bà Chiểu", "BV Gia Định", "Bình Thạnh"`. Mô hình đã học được các địa danh trong phần mô tả để kết nối các căn nhà gần nhau.
  * Đặc điểm kiến trúc: Mô hình bắt được các từ khóa chuyên sâu như `"nở hậu", "sân thượng", "hẻm xe hơi/xe tải"` để đưa vào danh sách ưu tiên.
  * Độ tương đồng: Thấp hơn căn `id=1`, chỉ dao động từ 25% đến 35%. Điều này thường phụ thuộc vào dữ liệu thu thập được, cũng như có thể do phần mô tả của các người bán khá dài, gây nhiễu dù đã xử lý.

## **Lưu ma trận kết quả consin và đọc lên khi cần đề xuất**

In [42]:
# Save cosine_sim to file
joblib.dump(cosine_sim, "models/nha_cosine_sim.pkl")

['models/nha_cosine_sim.pkl']

In [43]:
# # Open and read file to cosine_sim_new
# with open('nha_cosine_sim.pkl', 'rb') as f:
#     cosine_sim_new = pickle.load(f)

#

---



# **Gensim**

######
***Quy trình Gensim (5 bước)***
* Tiền xử lý: Tách từ (Tokenizing) và làm sạch văn bản (loại bỏ nhiễu).

* Tạo Từ điển (Dictionary): Ánh xạ từ ngữ sang ID số nguyên.

* Xây dựng Corpus (BoW): Biến tài liệu thành các vector tần suất.

* Huấn luyện TF-IDF: Gán trọng số cho các từ đặc trưng, loại bỏ từ phổ biến.

* Lập Chỉ mục (Similarity Index): So sánh và truy xuất các kết quả tương đồng nhất.

## **Preparation**

In [44]:
df_1.head(10)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [45]:
# Tokenize(split) the sentences into words
content_gem = [[text for text in x.split()] for x in df_1.Content_wt]

In [46]:
content_gem[:1]

[['Nhà_ngõ',
  ',',
  'hẻm',
  'Nội_thất',
  'đầy_đủ',
  'Nhà',
  'nở_hậu',
  'vân',
  'kiều',
  'an_cư',
  'nhà',
  'lê',
  'quang',
  'định',
  'p5',
  'giáp',
  'quận',
  '1',
  'vài',
  'bước',
  'ra',
  'hẻm',
  'ô_tô',
  'gần',
  'mặt_tiền',
  'khu',
  'dân_trí',
  'cao',
  'pháp_lý',
  'chuẩn',
  'giá',
  'dưới',
  '4',
  'tỷ',
  'hiếm',
  'có',
  '4.5',
  'm',
  'x',
  '8m',
  '2',
  'tầng',
  'chỉ',
  '3.85',
  'tỷ',
  'vị_trí',
  ':',
  'đường',
  'lê',
  'quang',
  'định',
  ',',
  'phường',
  '5',
  ',',
  'quận',
  'bình',
  'thạnh',
  'giáp',
  'quận',
  '1',
  ',',
  'di_chuyển',
  'cực',
  'nhanh',
  'vào',
  'trung_tâm',
  'gần',
  'bệnh_viện',
  'ung_bướu',
  ',',
  'bệnh_viện',
  'gia',
  'định',
  ',',
  'sân_bay',
  ',',
  'các',
  'trường',
  'đại_học',
  'lớn',
  'kết_cấu',
  ':',
  '1',
  'trệt',
  '1',
  'lầu',
  '2',
  'phòng',
  'ngủ',
  'master',
  '2',
  'wc',
  'thiết_kế',
  'rộng_rãi',
  'thoáng',
  'mát',
  'hẻm',
  ':',
  'cách',
  'hẻm',
  'ô_tô',
  'v

In [47]:
# hàm tiền xử lý
def preprocess_text_for_gensim(tokenized_text, stop_words):
    # Remove special characters and convert to lowercase
    cleaned_text = [t.lower() for t in tokenized_text if t not in ['', ' ', ',', '.', '...', '-',':', ';', '?', '%', '(', ')', '+', '/', "'", '&', '#', '*', 'hhnd85','hhhht012']]
    # Remove stop words
    cleaned_text = [t for t in cleaned_text if t not in stop_words]
    return cleaned_text


In [48]:
# Áp dụng cho từng danh sách từ trong content_gem
content_gem_re = [preprocess_text_for_gensim(text, stop_words) for text in content_gem]

In [49]:
df_1['content_gensim_processed'] = content_gem_re
df_1[['Content_wt', 'content_gensim_processed']].head(20)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [50]:
# Obtain the number of features based on dictionary: Use corpora.Dictionary
dictionary = corpora.Dictionary(content_gem_re)

In [51]:
# List of features in dictionary
dictionary.token2id

{'1': 0,
 '2': 1,
 '3.85': 2,
 '4': 3,
 '4.5': 4,
 '5': 5,
 '8m': 6,
 'an_cư': 7,
 'an_ninh': 8,
 'bình': 9,
 'bệnh_viện': 10,
 'chuẩn': 11,
 'chân': 12,
 'công_chứng': 13,
 'cực': 14,
 'di_chuyển': 15,
 'dân_trí': 16,
 'gia': 17,
 'giá': 18,
 'giáp': 19,
 'hiếm': 20,
 'hoàn_công': 21,
 'hẻm': 22,
 'hồng': 23,
 'khu': 24,
 'kiều': 25,
 'kết_cấu': 26,
 'lê': 27,
 'lầu': 28,
 'm': 29,
 'master': 30,
 'mát': 31,
 'mặt_tiền': 32,
 'ngủ': 33,
 'nhà_ngõ': 34,
 'nội_thất': 35,
 'nở_hậu': 36,
 'p5': 37,
 'pháp_lý': 38,
 'phòng': 39,
 'phường': 40,
 'quang': 41,
 'rộng_rãi': 42,
 'sân_bay': 43,
 'sạch_sẽ': 44,
 'sổ': 45,
 'thiết_kế': 46,
 'thoáng': 47,
 'thạnh': 48,
 'trung_tâm': 49,
 'trường': 50,
 'tầng': 51,
 'tỷ': 52,
 'ung_bướu': 53,
 'vân': 54,
 'wc': 55,
 'x': 56,
 'ô_tô': 57,
 'đường': 58,
 'đại_học': 59,
 'đầy_đủ': 60,
 'định': 61,
 '4pn': 62,
 '62m2': 63,
 '9': 64,
 'bình_thạnh': 65,
 'bấm': 66,
 'chủ': 67,
 'fuill': 68,
 'gửi': 69,
 'hcm': 70,
 'hàng_xóm': 71,
 'hình': 72,
 'keng': 7

In [52]:
# Numbers of features (word) in dictionary
feature_cnt = len(dictionary.token2id)
feature_cnt

10737

In [53]:
# Obtain corpus based on dictionary (dense matrix)
corpus = [dictionary.doc2bow(text) for text in content_gem_re]

In [54]:
corpus[1] # id, so lan xuat hien cua token trong van ban/ san pham

[(3, 1),
 (9, 1),
 (16, 1),
 (19, 1),
 (22, 1),
 (24, 1),
 (33, 1),
 (34, 1),
 (35, 1),
 (43, 1),
 (49, 1),
 (51, 1),
 (62, 1),
 (63, 1),
 (64, 1),
 (65, 1),
 (66, 1),
 (67, 1),
 (68, 1),
 (69, 1),
 (70, 1),
 (71, 1),
 (72, 1),
 (73, 1),
 (74, 1),
 (75, 1),
 (76, 1),
 (77, 1),
 (78, 1),
 (79, 1),
 (80, 1),
 (81, 1),
 (82, 1),
 (83, 1),
 (84, 1),
 (85, 1),
 (86, 1),
 (87, 1),
 (88, 1),
 (89, 1),
 (90, 1),
 (91, 1),
 (92, 1),
 (93, 1)]

## **Train the model**

In [55]:
# Use TF-IDF Model to process corpus, obtaining index
tfidf = models.TfidfModel(corpus)
# tính toán sự tương tự trong ma trận thưa thớt
index = similarities.SparseMatrixSimilarity(tfidf[corpus],
                                            num_features = feature_cnt)
# ma tran: n x n

In [56]:
df_gensim=pd.DataFrame(index)

In [57]:
# df_gensim

## **Hàm đề xuất nhà**

In [58]:
def display_recommendations_gensim(house_id, sim_matrix, df, top_n=5):

    # 1. Kiểm tra ID tồn tại
    if house_id not in df['id'].values:
        print(f"ID {house_id} không tồn tại.")
        return

    # 2. Tạo bản sao ma trận với index là ID
    sim_df = pd.DataFrame(sim_matrix)
    sim_df.index = df['id']
    sim_df.columns = df['id']

    # 3. Truy xuất dòng tương đồng bằng ID
    series_sim = sim_df.loc[house_id].drop(house_id, errors='ignore')

    # 4. Lấy Top N
    top_results = series_sim.nlargest(top_n)
    top_ids = top_results.index.tolist()

    # 5. Hiển thị thông tin nhà hiện tại
    print(f"--- ĐANG XEM: ID {house_id} ---")
    display(df[df['id'] == house_id][['id', 'quan', 'mo_ta_clean']])

    print("\n" + "-"*50 + f"TOP {top_n} GỢI Ý TƯƠNG TỰ (Gensim Method)" + "-"*50)

    # 6. Tạo DataFrame kết quả và xử lý hiển thị phần trăm
    recs = df[df['id'].isin(top_ids)].copy()
    
    # Tính giá trị số trước để sắp xếp
    recs['score_num'] = recs['id'].map(top_results) * 100
    recs = recs.sort_values(by='score_num', ascending=False)

    # --- ĐÂY LÀ KHÚC HIỆN PHẦN TRĂM ---
    recs['Độ tương đồng'] = recs['score_num'].apply(lambda x: f"{x:.2f}%")

    # Chỉ lấy các cột cần thiết để hiển thị
    recs_final = recs[['id', 'quan', 'mo_ta_clean', 'Độ tương đồng']]
    recs_final.columns = ['ID', 'Quận', 'Mô tả', 'Độ tương đồng']

    display(recs_final.reset_index(drop=True))

## **Testing**

In [59]:
# Chạy thử nghiệm với id = 1
display_recommendations_gensim(house_id=1, sim_matrix=df_gensim, df=df_1)

--- ĐANG XEM: ID 1 ---


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



--------------------------------------------------TOP 5 GỢI Ý TƯƠNG TỰ (Gensim Method)--------------------------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [60]:
# Chạy thử nghiệm với id = 10
display_recommendations_gensim(house_id=10, sim_matrix=df_gensim, df=df_1)

--- ĐANG XEM: ID 10 ---


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



--------------------------------------------------TOP 5 GỢI Ý TƯƠNG TỰ (Gensim Method)--------------------------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


#### *Nhìn chung thì kết quả chả khác bên Cosine là bao, chỉ thấp hơn một chút về similarity thôi. Cũng phải do công thức tính của hai đứa là khác nhau*

#


---



# ***Trường hợp:*** **Khách hàng nhập thông tin tìm kiếm**

In [61]:
def search_house_cosine(query, vectorizer, tfidf_matrix, df, top_n=5):
    # 1. Tiền xử lý tối ưu: Làm sạch HTML, icon và chuẩn hóa text tương tự như tập huấn luyện
    cleaned_query = clean_html_icon(query)

    # 2. Tách từ tiếng Việt
    query_wt = tokenize(cleaned_query)

    # 3. Chuyển đổi sang vector TF-IDF
    query_tfidf = vectorizer.transform([query_wt])

    # 4. Tính toán Cosine Similarity
    sim_scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()

    # 5. Lọc và lấy Top N kết quả
    top_indices = np.argsort(sim_scores)[-top_n:][::-1]
    # Lọc bỏ các kết quả có độ tương đồng bằng 0
    top_indices = [i for i in top_indices if sim_scores[i] > 0]

    if not top_indices:
        print(f"Không tìm thấy kết quả nào phù hợp với truy vấn: '{query}'")
        return

    print(f"--- TOP {len(top_indices)} KẾT QUẢ GỢI Ý TỐI ƯU CHO: '{cleaned_query}' ---")

    # 6. Tạo DataFrame kết quả (Đã bỏ Styler)
    results = df.iloc[top_indices][['id', 'quan', 'mo_ta_clean']].copy()
    
    # Tính toán giá trị phần trăm và gán vào cột mới
    results['Độ tương đồng'] = [f"{s * 100:.2f}%" for s in sim_scores[top_indices]]

    # Đổi tên cột sang tiếng Việt
    results.columns = ['ID', 'Quận', 'Mô tả', 'Độ tương đồng']

    # Hiển thị bảng thuần túy (Mặc định của VS Code/Jupyter)
    display(results.reset_index(drop=True))

In [62]:
user_query_test = "trung tâm, phú nhuận, có bãi đỗ xe, có sân thượng, gần bệnh viện, đại học"
search_house_cosine(user_query_test, vectorizer, tfidf_matrix, df_1, top_n=5)

--- TOP 5 KẾT QUẢ GỢI Ý TỐI ƯU CHO: 'trung tâm, phú nhuận, có bãi đỗ xe, có sân thượng, gần bệnh viện, đại học' ---


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


#

---



# **Hybrid Recommender**

* Tính Content Similarity (từ Content based filtering: Dùng model `cosine_sim` đã lưu - cosine_smilarity)
* Tính Price Similarity: min-max scaler giá sau đó tính khoảng cách
* Tính Location Similarity: tính khoảng cách Quận (theo vector onehot encoder)

## *Functions preparation*

In [63]:
df_1.head(20)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [64]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import pairwise_distances

### *Price Similarity*

In [65]:
def compute_price_similarity(df, price_col='gia_ban_ty'):
    # Handle NaNs with median and scale prices
    prices = df[[price_col]].fillna(df[price_col].median())
    scaler = MinMaxScaler()
    prices_scaled = scaler.fit_transform(prices)

    # Compute Euclidean distance and convert to similarity (1 - distance)
    price_dist = pairwise_distances(prices_scaled, metric='euclidean')
    return 1 - price_dist

### *Location Similarity*

In [66]:
def compute_location_similarity(df, loc_col='quan'):
    # One-hot encode districts and compute cosine similarity
    loc_encoded = pd.get_dummies(df[loc_col]).astype(int)
    return cosine_similarity(loc_encoded)

### *Kết hợp trọng số*

In [67]:
def build_hybrid_matrix(content_sim, price_sim, loc_sim, w_content=0.5, w_price=0.25, w_loc=0.25):
    # Weighted combination of similarity matrices
    return (w_content * content_sim) + (w_price * price_sim) + (w_loc * loc_sim)

### *Thực thi trên data*

In [68]:
price_sim_mat = compute_price_similarity(df_1)
loc_sim_mat = compute_location_similarity(df_1)

hybrid_sim_matrix = build_hybrid_matrix(
    cosine_sim, price_sim_mat, loc_sim_mat,
    w_content=0.5, w_price=0.25, w_loc=0.25
)

## ***Hàm đề xuất nhà***

In [69]:
def get_hybrid_recommendations(house_id, df, hybrid_matrix, top_n=5):
    # 1. Check if ID exists
    if house_id not in df['id'].values:
        print(f"ID {house_id} không tồn tại.")
        return None

    # 2. Get the index of the target house
    idx = df.index[df['id'] == house_id][0]

    # 3. Retrieve similarity scores for this house
    sim_scores = list(enumerate(hybrid_matrix[idx]))

    # 4. Sort and exclude the item itself
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    # 5. Extract indices and scores
    indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    # 6. Display target house information (Bỏ Styler)
    print(f"\n{'-'*40} ĐANG XEM NHÀ (ID: {house_id}) {'-'*40}")
    target_house = df.iloc[[idx]][['id', 'quan', 'phuong_val', 'gia_ban_ty', 'mo_ta_clean']].copy()
    
    # Format giá bán trực tiếp
    target_house['gia_ban_ty'] = target_house['gia_ban_ty'].apply(lambda x: f"{x:.2f} tỷ")
    target_house.columns = ['ID', 'Quận', 'Phường', 'Giá', 'Mô tả']
    display(target_house)

    print("\n" + "-"*30 + " TOP GỢI Ý HYBRID (Text + Price + Location) " + "-"*30)

    # 7. Create results DataFrame
    results = df.iloc[indices][['id', 'quan', 'phuong_val', 'gia_ban_ty', 'mo_ta_clean']].copy()
    
    # Chuyển đổi score và giá sang dạng chuỗi kèm đơn vị
    results['Độ tương đồng'] = [f"{s * 100:.2f}%" for s in scores]
    results['gia_ban_ty'] = results['gia_ban_ty'].apply(lambda x: f"{x:.2f} tỷ")

    # Chọn và đổi tên cột sang tiếng Việt
    results_final = results[['id', 'quan', 'phuong_val', 'gia_ban_ty', 'mo_ta_clean', 'Độ tương đồng']]
    results_final.columns = ['ID', 'Quận', 'Phường', 'Giá (Tỷ đồng)', 'Mô tả', 'Độ tương đồng']

    # 8. Trả về DataFrame thuần thay vì Styler
    return results_final.reset_index(drop=True)

## **Testing**

In [70]:
# ID 1
recommendations = get_hybrid_recommendations(1, df_1, hybrid_sim_matrix)
recommendations


---------------------------------------- ĐANG XEM NHÀ (ID: 1) ----------------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



------------------------------ TOP GỢI Ý HYBRID (Text + Price + Location) ------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [71]:
# ID 18
recommendations = get_hybrid_recommendations(18, df_1, hybrid_sim_matrix)
recommendations


---------------------------------------- ĐANG XEM NHÀ (ID: 18) ----------------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)



------------------------------ TOP GỢI Ý HYBRID (Text + Price + Location) ------------------------------


Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


***Nhận xét về Hybrid Recommender:***

*   **Sự cân bằng:** Hệ thống đã kết hợp tốt cả 3 yếu tố. Các kết quả gợi ý giờ đây không chỉ giống về mô tả (text) mà còn nằm trong cùng một phân khúc giá và cùng khu vực địa lý (Quận).
*   **Độ thực tế:** Việc thêm trọng số cho `Price` và `Location` giúp danh sách gợi ý trở nên hữu ích hơn với người dùng, tránh trường hợp gợi ý một căn nhà rất giống về kiến trúc nhưng lại ở quá xa hoặc có giá quá cao so với căn đang xem.
*   **Tính linh hoạt:** Chúng ta có thể dễ dàng điều chỉnh các trọng số `w_content`, `w_price`, `w_loc` tùy theo chiến lược kinh doanh (ví dụ: ưu tiên gợi ý nhà cùng khu vực hoặc ưu tiên nhà cùng tầm giá).

In [72]:
# Note